# LLM Fine-tuning（Unsloth）

## 共通の事前準備（必ず）

- `training/params.yaml` の **`hf_lora_repo`** を、自分の Hugging Face 上のモデル用リポジトリ名（例: `ユーザー名/リポジトリ名`）に編集し、**GitHub に push** しておく。
- フォークして使う場合は、セクション 2 の **`REPO_OWNER` / `REPO_NAME`** も自分用に直してから push する。

## Google Colab で動かす場合（推奨）

1. （初回のみ）左メニュー **🔑 シークレット** に **`HF_TOKEN`**（Hugging Face の Write トークン）を登録する。
2. **ランタイム → ランタイムのタイプを変更** で **GPU（T4 など）** を選ぶ。
3. **ランタイム → すべてのセルを実行**（または上から順に実行）。

プライベート GitHub を clone する場合のみ、シークレット **`GITHUB_TOKEN`** も追加する（**公開リポジトリなら不要**）。

## ローカル（VS Code / Jupyter）で動かす場合

- **推奨**: リポジトリのルートに **`.env`** を置き、`HF_TOKEN=hf_...` と書く（`.env.example` をコピーして作成）。**セクション 2 が自動で読み込みます**（`.env` は Git に含めません）。
- そのほか、環境変数や `%env HF_TOKEN=hf_...`、または**未設定ならセクション 2 が入力プロンプト**でも可。
- ワークスペースは **`llm-model-generation` のリポジトリルート**を開く（`training` フォルダと並ぶ階層）。
- セクション 2 は **git clone をスキップ**し、カレントディレクトリからルートを自動検出する。
- 学習（Unsloth・4bit）は **NVIDIA GPU があるマシン**向け。GPU が無い場合は **Colab を使う**こと。


## 1. 依存パッケージのインストール


In [ ]:
# Install Unsloth and other dependencies
!pip install -q "unsloth[colab-new]" peft accelerate bitsandbytes transformers trl huggingface_hub pyyaml datasets python-dotenv


## 2. シークレット・リポジトリのクローン（またはローカルで作業ディレクトリを合わせる）

- **Google Colab**: `/content` に git clone し、`HF_TOKEN` は Colab の **シークレット**から読みます。
- **ローカル（VS Code / Jupyter）**: `/content` は存在しないため **clone は行いません**。カレントディレクトリからリポジトリのルートを自動検出し、見つかった **`/.env` を読み込んで `HF_TOKEN` を設定**します（`python-dotenv` を使用）。


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path
from typing import Optional


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_secret(name: str) -> Optional[str]:
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return None


def _find_repo_root(start: Path) -> Optional[Path]:
    p = start.resolve()
    for _ in range(12):
        if (p / "training" / "finetune_script.py").is_file():
            return p
        if p.parent == p:
            break
        p = p.parent
    return None


REPO_OWNER = "TsutsujiStudyTeam"
REPO_NAME = "llm-model-generation"

if _in_colab():
    content = Path("/content")
    content.mkdir(parents=True, exist_ok=True)
    repo_path = content / REPO_NAME
    if repo_path.exists():
        shutil.rmtree(repo_path, ignore_errors=True)
    os.chdir(content)
    github_token = _colab_secret("GITHUB_TOKEN")
    if github_token:
        clone_url = f"https://{github_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
    else:
        clone_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    subprocess.run(["git", "clone", clone_url, str(REPO_NAME)], check=True)
    os.chdir(repo_path)
else:
    root = _find_repo_root(Path.cwd())
    if root is None:
        raise RuntimeError(
            "リポジトリのルートを特定できませんでした。"
            "VS Code ではフォルダとして llm-model-generation を開き、ノートブックを開き直すか、"
            "カーネルのカレントディレクトリをリポジトリのルートにしてください。"
        )
    os.chdir(root)
    print("ローカル実行: git clone はスキップしました。")


def _load_dotenv_here() -> None:
    """カレントがリポジトリルートになったあとで .env を読む（ローカル用）。"""
    try:
        from dotenv import load_dotenv
    except ImportError:
        return
    env_path = Path.cwd() / ".env"
    if env_path.is_file():
        load_dotenv(env_path)
        print("（.env を読み込みました）")


_load_dotenv_here()

hf_token = (_colab_secret("HF_TOKEN") or os.environ.get("HF_TOKEN", "").strip())
if not hf_token and not _in_colab():
    import getpass

    print(
        "HF_TOKEN が未設定です。リポジトリのルートに .env を置き HF_TOKEN=... と記載するか（.env.example 参照）、"
        "このあと表示される入力に貼り付けてください（画面上には表示されません）。"
    )
    hf_token = getpass.getpass("HF_TOKEN: ").strip()
if not hf_token:
    raise RuntimeError(
        "HF_TOKEN がありません。\n"
        "・Google Colab: 左「🔑 シークレット」に名前 HF_TOKEN（値は Write トークン）を追加。\n"
        "・ローカル: リポジトリルートの .env、環境変数、または入力で渡す。"
    )
os.environ["HF_TOKEN"] = hf_token

print("Working directory:", os.getcwd())


## 3. 学習の実行（`finetune_script.py`）

対話入力はありません。`training/params.yaml` と環境変数 `HF_TOKEN`（および任意の `HF_LORA_REPO`）で学習します。


In [ ]:
# subprocess + check=True だと CalledProcessError ばかり目立ち、本体の Traceback が分かりにくい。
# IPython でスクリプトを直接実行すると、エラーがこのセルにそのまま出る（コメントの下でも動く）。
from IPython import get_ipython

get_ipython().run_line_magic("run", "training/finetune_script.py")


## 4. 簡易推論テスト

`params.yaml` の **ベースモデル（`hf_model_repo`）** と **Hub に push した LoRA（`hf_lora_repo`）** を組み合わせて読み込みます。`training/lora_adapter_merged` を `from_pretrained` だけで読むと、Unsloth × Transformers 5.x などで tokenizer が壊れたように見えるエラー（例: `KeyError: 'shape'`）が出ることがあるため、このセルでは使いません。

セクション 2 で `HF_TOKEN` が環境変数に入っている必要があります（プライベート LoRA の場合）。**CUDA（GPU）が必要**です。

**CPU の PC**: 下のコードセルは **例外を出さず** `[スキップ]` と表示して終わります。もし `RuntimeError: CUDA が利用できません` と出る場合、セルが**古い版のまま**です。**ディスク上の `finetune.ipynb` を開き直す**か、**カーネル再起動**後に、下のコードセルをもう一度実行してください（`raise` が無い最新版である必要があります）。


In [ ]:
# CPU のときは print のみ（raise しない）。このコメントが無い／raise があるなら古いセルです。
import yaml
from pathlib import Path

import torch

# Unsloth は import 時点で GPU を要求するため、先に CUDA の有無だけ判定する
if not torch.cuda.is_available():
    print(
        "[スキップ] CUDA がありません。簡易推論は GPU（例: Colab の T4）上でのみ実行されます。"
        "CPU の PC ではこのセルは何もしません（エラーではありません）。"
    )
else:
    import os
    from collections.abc import Mapping

    from unsloth import FastLanguageModel

    ROOT = Path.cwd()
    with open(ROOT / "training" / "params.yaml", encoding="utf-8") as f:
        params = yaml.safe_load(f)

    max_seq_length = int(params["max_seq_length"])
    base_id = str(params["hf_model_repo"])
    lora_id = str(params["hf_lora_repo"]).strip()

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=base_id,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.load_lora_into_model(
        model,
        lora_id,
        token=os.environ.get("HF_TOKEN"),
    )
    FastLanguageModel.for_inference(model)

    messages = [
        {"role": "user", "content": "私は犬が好きです。これを否定文に変換してください。"},
    ]
    batch = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    if isinstance(batch, torch.Tensor):
        batch = batch.to(model.device)
        outputs = model.generate(batch, max_new_tokens=64, use_cache=True)
    elif isinstance(batch, Mapping):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model.generate(**batch, max_new_tokens=64, use_cache=True)
    else:
        batch = batch.to(model.device)
        outputs = model.generate(batch, max_new_tokens=64, use_cache=True)

    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
